#### Imports

#### Imports

In [1]:
import stim
import pymatching
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch_geometric.data import Data

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# If CUDA is available, show more details
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("PyTorch is using CPU only")

PyTorch is using CPU only


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


#### Definitions

##### Basic Stim & MWPM Definitions

In [4]:
def surface_code_circuit(p, d): # physical error rate, distance
  return stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=d,
    distance=d,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
    before_measure_flip_probability=p,
    before_round_data_depolarization=p)

def count_logical_errors(circuit: stim.Circuit, num_shots: int) -> int:
  # Sample the circuit.
  sampler = circuit.compile_detector_sampler()
  detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

  # Configure a decoder using the circuit.
  detector_error_model = circuit.detector_error_model(decompose_errors=True)
  matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

  # Run the decoder.
  predictions = matcher.decode_batch(detection_events)

  # basically compare predictions with observable_flips (what we should have measured)

  # Count the mistakes.
  num_errors = 0
  for shot in range(num_shots):
    actual_for_shot = observable_flips[shot]
    predicted_for_shot = predictions[shot]
    if not np.array_equal(actual_for_shot, predicted_for_shot):
        num_errors += 1
  return num_errors

def ler_mwpm(p, d): # logical error rate, minimum weight perfect matching
  num_shots = 100000
  circuit = surface_code_circuit(p, d)
  num_errors = count_logical_errors(circuit, num_shots)

  return num_errors / num_shots

def plot_mwpm():
  num_shots = 100000
  for d in [3, 5, 7]:
    xs = []
    ys = []
    yerrs = []
    for noise in np.linspace(0.001, 0.008, 8):
      ler = ler_mwpm(noise, d)
      xs.append(noise)
      ys.append(ler)
      yerrs.append(np.sqrt(ler * (1 - ler) / num_shots))
  plt.loglog()
  plt.xlabel("physical error rate")
  plt.ylabel("logical error rate per shot")
  plt.legend()
  plt.show()


##### GNN Trainer Class

In [5]:
class SurfaceCodeSampler:
    """
    Sampler class for surface code quantum error correction.
    
    This class generates surface code circuits and creates training datasets
    with configurable error rates. 
    
    Attributes:
        d (int): Code distance
        default_p (float): Default physical error rate
        circuit (stim.Circuit): The generated surface code circuit at default_p
        device (torch.device): Device to use for tensors
    """
    
    def __init__(self, d: int, p: float = 0.01, device: torch.device = None):
        """
        Initialize the sampler with a surface code circuit.
        
        Args:
            d (int): Code distance (determines code size and rounds)
            p (float): Default physical error rate (used if not overridden)
            device (torch.device): Device for tensors (defaults to CUDA if available)
        """
        self.d = d
        self.default_p = p
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Generate and save the default circuit
        self.circuit = self._generate_circuit(p)
        
        # Cache circuits for different p values to avoid regenerating
        self._circuit_cache = {p: self.circuit}
        
        print(f"SurfaceCodeSampler initialized:")
        print(f"  Distance: {d}")
        print(f"  Default error rate: {p}")
        print(f"  Device: {self.device}")
        print(f"  Num detectors: {self.circuit.num_detectors}")
    
    def _generate_circuit(self, p: float) -> stim.Circuit:
        """Generate a surface code circuit with given error rate."""
        return stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            rounds=self.d,
            distance=self.d,
            after_clifford_depolarization=p,
            after_reset_flip_probability=p,
            before_measure_flip_probability=p,
            before_round_data_depolarization=p
        )
    
    def _get_circuit(self, p: float) -> stim.Circuit:
        """Get circuit for a given p, using cache if available."""
        if p not in self._circuit_cache:
            self._circuit_cache[p] = self._generate_circuit(p)
        return self._circuit_cache[p]
    
    def get_circuit(self) -> stim.Circuit:
        """Return the default circuit."""
        return self.circuit
    
    def sample(self, 
               num_samples: int,
               p_values: list[float] = None,
               p_weights: list[float] = None,
               return_p_labels: bool = False) -> tuple:
        """
        Generate training data samples with configurable error rate distribution.
        
        This function generates syndrome detection data and observable flip labels.
        You can specify multiple error rates with weights to control what percentage
        of the dataset uses each error rate.
        
        Args:
            num_samples (int): Total number of samples to generate
            p_values (list[float], optional): Array of physical error rates to use.
                                              Defaults to [self.default_p].
            p_weights (list[float], optional): Array of weights (same length as p_values).
                                               Must sum to 1.0. Determines what fraction
                                               of samples use each error rate.
                                               Defaults to [1.0] (all samples at one rate).
            return_p_labels (bool): If True, also return which p was used for each sample.
        
        Returns:
            tuple: (detections, labels) or (detections, labels, p_indices) if return_p_labels
                - detections: torch.Tensor [num_samples, num_detectors] syndrome measurements (-1 or +1)
                - labels: torch.Tensor [num_samples] observable flip labels (0 or 1)
                - p_indices: torch.Tensor [num_samples] index into p_values for each sample
        
        Examples:
            # Single error rate (uses default p)
            detections, labels = sampler.sample(10000)
            
            # Single custom error rate
            detections, labels = sampler.sample(10000, p_values=[0.005], p_weights=[1.0])
            
            # Mixed error rates: 50% at p=0.001, 30% at p=0.003, 20% at p=0.005
            detections, labels = sampler.sample(
                10000,
                p_values=[0.001, 0.003, 0.005],
                p_weights=[0.5, 0.3, 0.2]
            )
        """
        # Handle defaults
        if p_values is None:
            p_values = [self.default_p]
        if p_weights is None:
            p_weights = [1.0]
        
        # Validate inputs
        if len(p_values) != len(p_weights):
            raise ValueError(f"p_values and p_weights must have same length. "
                           f"Got {len(p_values)} and {len(p_weights)}")
        
        weight_sum = sum(p_weights)
        if not np.isclose(weight_sum, 1.0, atol=1e-6):
            raise ValueError(f"p_weights must sum to 1.0, got {weight_sum}")
        
        # Calculate samples per error rate
        samples_per_p = []
        remaining = num_samples
        for i, weight in enumerate(p_weights):
            if i == len(p_weights) - 1:
                # Last one gets remaining to handle rounding
                n = remaining
            else:
                n = int(round(num_samples * weight))
                remaining -= n
            samples_per_p.append(n)
        
        # Generate samples for each error rate
        all_detections = []
        all_labels = []
        all_p_indices = []
        
        for i, (p, n) in enumerate(zip(p_values, samples_per_p)):
            if n <= 0:
                continue
                
            circuit = self._get_circuit(p)
            sampler = circuit.compile_detector_sampler()
            detections, flips = sampler.sample(n, separate_observables=True)
            
            # Convert to tensors
            # Convert detections: 0 -> -1, 1 -> +1 for easier reading
            det_np = detections.astype(np.float32) * 2 - 1
            det_tensor = torch.from_numpy(det_np)
            label_tensor = torch.from_numpy(flips.astype(np.float32).flatten())
            
            all_detections.append(det_tensor)
            all_labels.append(label_tensor)
            all_p_indices.append(torch.full((n,), i, dtype=torch.long))
        
        # Concatenate all samples
        detections = torch.cat(all_detections, dim=0).to(self.device)
        labels = torch.cat(all_labels, dim=0).to(self.device)
        p_indices = torch.cat(all_p_indices, dim=0).to(self.device)
        
        # Shuffle the dataset so error rates are mixed
        perm = torch.randperm(num_samples, device=self.device)
        detections = detections[perm]
        labels = labels[perm]
        p_indices = p_indices[perm]
        
        if return_p_labels:
            return detections, labels, p_indices
        return detections, labels

#### GNN Approaches

##### Regular Graphs

In [6]:
# TODO if interesting, I want to find out about sparse graphs

##### Sparse Graphs

In [7]:
class SparseGraph:
    """
    Converts syndrome detections into sparse PyTorch Geometric graphs.
    
    Following the paper's representation (Figure 3.1):
    - Only fired detectors become nodes (sparse graph)
    - Node features: [X?, Z?, d_North, d_West, d_time] (5 features)
    - Edge weights: e_ij = (max{|d_North_i - d_North_j|, |d_West_i - d_West_j|, |d_time_i - d_time_j|})^(-2)
    - K-nearest neighbor connectivity for scalability
    
    This class dynamically handles detections of any size by inferring the code
    distance from the number of detectors and generating coordinates on-the-fly.
    Supports variable-size batches where each sample may come from a different distance.
    
    Attributes:
        k_neighbors (int): Maximum number of neighbors per node
        device (torch.device): Device for output tensors
    """
    
    def __init__(self, k_neighbors: int = 6, device: torch.device = None):
        """
        Initialize the SparseGraph builder.
        
        Args:
            k_neighbors (int): Maximum number of neighbors per node (default 6)
            device (torch.device): Device for output tensors
        """
        self.k_neighbors = k_neighbors
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Caches for dynamic coordinate/feature computation
        self._coord_cache = {}    # num_detectors -> detector_coords dict
        self._feature_cache = {}  # num_detectors -> (all_features, normalization_bounds)
        
        print(f"SparseGraph initialized:")
        print(f"  K neighbors: {k_neighbors}")
        print(f"  Device: {self.device}")
        print(f"  Mode: Dynamic (supports any code distance)")
    
    @staticmethod
    def _infer_distance(num_detectors: int) -> int:
        """
        Infer code distance from number of detectors.
        
        For rotated surface code with rounds=d:
        - d=3: 8 stabilizers * 3 rounds = 24 detectors
        - d=5: 24 stabilizers * 5 rounds = 120 detectors
        - d=7: 48 stabilizers * 7 rounds = 336 detectors
        """
        # Try known mappings first
        known = {24: 3, 120: 5, 336: 7, 680: 9, 1168: 11}
        if num_detectors in known:
            return known[num_detectors]
        
        # Otherwise solve: num_stabilizers * d = num_detectors
        # where num_stabilizers = d^2 - 1 for rotated surface code
        # So: (d^2 - 1) * d = num_detectors
        for d in range(3, 50, 2):  # odd distances only
            if (d * d - 1) * d == num_detectors:
                return d
        
        raise ValueError(f"Cannot infer distance from {num_detectors} detectors")
    
    @staticmethod
    def _generate_detector_coords(distance: int) -> dict:
        """
        Generate detector coordinates for a standard rotated surface code.
        
        For a rotated surface code of distance d with d rounds:
        - Stabilizers are arranged on a rotated grid
        - X and Z stabilizers alternate in a checkerboard pattern
        - Each round creates one detector per stabilizer
        
        Args:
            distance: The code distance (must be odd >= 3)
            
        Returns:
            dict mapping detector_id -> [x, y, t, basis]
            where basis is 0 for Z-stabilizer, 1 for X-stabilizer
        """
        d = distance
        num_stabilizers = d * d - 1
        
        # Generate stabilizer positions for rotated surface code
        # In Stim's convention, stabilizers are at positions where x+y is even
        # Grid spans from (1,1) to (2d-1, 2d-1) with step 2
        stabilizers = []
        
        for y in range(1, 2 * d, 2):
            for x in range(1, 2 * d, 2):
                # Check if this is a valid stabilizer position
                # For rotated surface code, we exclude certain corner positions
                # The pattern depends on the specific layout
                
                # Standard check: position must have (x+y)/2 even or odd for X/Z
                # But we also need to ensure we're within the rotated diamond
                
                # Simpler approach: enumerate all valid stabilizer positions
                # For a d x d rotated code, stabilizers are at positions
                # where the Manhattan distance from center is appropriate
                
                stabilizers.append((x, y))
        
        # Filter to get exactly d^2-1 stabilizers (remove corners as needed)
        # For rotated surface code: stabilizers are at (x,y) where both are odd
        # and form the interior + edges of the rotated square
        
        # More accurate generation based on Stim's layout:
        # Stabilizers at (x, y) where x, y are both odd, ranging 1 to 2d-1
        # This gives d^2 positions, we remove 1 (typically a corner or it's implicit)
        
        # Actually for rotated surface code, the number of stabilizers is exactly d^2-1
        # because we have d^2 data qubits and d^2-1 independent stabilizers
        
        # Generate all positions where x,y are odd in range [1, 2d-1]
        all_positions = []
        for y in range(1, 2 * d, 2):
            for x in range(1, 2 * d, 2):
                all_positions.append((x, y))
        
        # This gives d^2 positions. For surface code, all are valid stabilizers
        # except the logical operators reduce independence by 1
        # Stim includes all d^2-1 physical stabilizer measurements
        
        # Use the first d^2-1 positions in row-major order
        stabilizers = all_positions[:num_stabilizers]
        
        # If we got more positions than needed (shouldn't happen for correct d),
        # or fewer, adjust
        if len(all_positions) == d * d:
            # Standard case: take all but exclude one (typically Stim handles this)
            # For now, use all d^2-1 stabilizers
            stabilizers = all_positions[:num_stabilizers]
        
        # Assign basis: checkerboard pattern based on (x+y)/2 mod 2
        # Convention: (x+y)/2 even -> Z stabilizer (basis=0)
        #             (x+y)/2 odd  -> X stabilizer (basis=1)
        
        detector_coords = {}
        det_id = 0
        
        for t in range(d):  # d rounds of measurement
            for (x, y) in stabilizers:
                basis = int(((x + y) // 2) % 2)  # 0 for Z, 1 for X
                detector_coords[det_id] = [float(x), float(y), float(t), float(basis)]
                det_id += 1
        
        return detector_coords
    
    def _get_coords_and_features(self, num_detectors: int):
        """
        Get detector coordinates and precomputed features for a given number of detectors.
        Uses caching to avoid recomputation.
        
        Args:
            num_detectors: Number of detectors in the detection sample
            
        Returns:
            tuple: (detector_coords, all_features, normalization_bounds)
        """
        if num_detectors in self._feature_cache:
            return self._feature_cache[num_detectors]
        
        # Cache miss: compute everything
        distance = self._infer_distance(num_detectors)
        
        # Generate or retrieve coordinates
        if num_detectors not in self._coord_cache:
            self._coord_cache[num_detectors] = self._generate_detector_coords(distance)
        
        detector_coords = self._coord_cache[num_detectors]
        
        # Compute normalization bounds
        coords = list(detector_coords.values())
        x_vals = [c[0] for c in coords]
        y_vals = [c[1] for c in coords]
        t_vals = [c[2] for c in coords]
        
        x_min, x_max = min(x_vals), max(x_vals)
        y_min, y_max = min(y_vals), max(y_vals)
        t_min, t_max = min(t_vals), max(t_vals)
        
        norm_bounds = {
            'x_min': x_min, 'x_max': x_max,
            'y_min': y_min, 'y_max': y_max,
            't_min': t_min, 't_max': t_max
        }
        
        # Precompute features for all detectors
        features = []
        for det_id in range(num_detectors):
            coord = detector_coords.get(det_id, [0, 0, 0, 0])
            x, y, t = coord[0], coord[1], coord[2]
            
            # Get basis from 4th coordinate
            if len(coord) >= 4:
                b = coord[3]
                is_x = 1.0 if b == 1 else 0.0
                is_z = 1.0 if b == 0 else 0.0
            else:
                # Fallback: checkerboard pattern
                stabilizer_type = int(((x + y) / 2) % 2)
                is_x = float(stabilizer_type)
                is_z = 1.0 - is_x
            
            # Normalized distances (0 to 1)
            d_west = (x - x_min) / max(1, x_max - x_min)
            d_north = (y - y_min) / max(1, y_max - y_min)
            d_time = (t - t_min) / max(1, t_max - t_min)
            
            features.append([is_x, is_z, d_north, d_west, d_time])
        
        all_features = torch.tensor(features, dtype=torch.float32)
        
        # Cache the result
        self._feature_cache[num_detectors] = (detector_coords, all_features, norm_bounds)
        
        return detector_coords, all_features, norm_bounds
    
    def _supremum_distance(self, feat_i: torch.Tensor, feat_j: torch.Tensor) -> float:
        """
        Compute supremum norm distance between two nodes.
        Uses d_North, d_West, d_time (indices 2, 3, 4 of features).
        """
        # Features are [X?, Z?, d_North, d_West, d_time]
        d_north_diff = abs(feat_i[2] - feat_j[2])
        d_west_diff = abs(feat_i[3] - feat_j[3])
        d_time_diff = abs(feat_i[4] - feat_j[4])
        
        return max(d_north_diff.item(), d_west_diff.item(), d_time_diff.item())
    
    def _compute_edge_weight(self, sup_dist: float) -> float:
        """
        Compute edge weight from supremum distance.
        e_ij = (max_distance)^(-2)
        """
        if sup_dist < 1e-9:
            return 1.0  # Same position, max weight
        return sup_dist ** (-2)
    
    def to_pyg(self, detections: torch.Tensor, label: torch.Tensor) -> Data:
        """
        Convert a single detection sample to a PyTorch Geometric Data object.
        
        Args:
            detections: Tensor of shape [num_detectors] with values -1 or +1
            label: Scalar tensor (0 or 1) for the observable flip
            
        Returns:
            torch_geometric.data.Data with:
                - x: Node features [num_fired, 5]
                - edge_index: Edge connectivity [2, num_edges]
                - edge_attr: Edge weights [num_edges, 1]
                - y: Label
        """
        # Move to CPU for processing
        detections_cpu = detections.cpu()
        num_detectors = detections_cpu.shape[0]
        
        # Get cached coordinates and features for this detection size
        _, all_features, _ = self._get_coords_and_features(num_detectors)
        
        # Find fired detectors (value == +1)
        fired_mask = detections_cpu > 0
        fired_indices = torch.where(fired_mask)[0].tolist()
        
        # Handle edge case: no fired detectors
        if len(fired_indices) == 0:
            return Data(
                x=torch.zeros((0, 5), dtype=torch.float32),
                edge_index=torch.zeros((2, 0), dtype=torch.long),
                edge_attr=torch.zeros((0, 1), dtype=torch.float32),
                y=label.cpu().clone()
            )
        
        # Handle edge case: only one fired detector
        if len(fired_indices) == 1:
            node_features = all_features[fired_indices].unsqueeze(0)
            return Data(
                x=node_features,
                edge_index=torch.zeros((2, 0), dtype=torch.long),
                edge_attr=torch.zeros((0, 1), dtype=torch.float32),
                y=label.cpu().clone()
            )
        
        # Get node features for fired detectors
        node_features = all_features[fired_indices]
        num_nodes = len(fired_indices)
        
        # Build edges using k-nearest neighbors based on supremum distance
        edges = []
        edge_weights = []
        
        # Compute pairwise supremum distances
        for i in range(num_nodes):
            distances = []
            for j in range(num_nodes):
                if i != j:
                    sup_dist = self._supremum_distance(node_features[i], node_features[j])
                    distances.append((j, sup_dist))
            
            # Sort by distance and take k nearest
            distances.sort(key=lambda x: x[1])
            k = min(self.k_neighbors, len(distances))
            
            for j, sup_dist in distances[:k]:
                edges.append([i, j])
                edge_weights.append(self._compute_edge_weight(sup_dist))
        
        # Convert to tensors
        if len(edges) > 0:
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            edge_attr = torch.tensor(edge_weights, dtype=torch.float32).unsqueeze(1)
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr = torch.zeros((0, 1), dtype=torch.float32)
        
        return Data(
            x=node_features,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=label.cpu().clone()
        )
    
    def batch_to_pyg(self, detections: list, labels: list) -> list:
        """
        Convert a batch of detection samples to a list of PyG Data objects.
        Supports variable-size inputs where each detection may have different dimensions.
        
        Args:
            detections: List of tensors, each of shape [num_detectors_i]
            labels: List of scalar tensors
            
        Returns:
            List of torch_geometric.data.Data objects
        """
        return [self.to_pyg(det, lbl) for det, lbl in zip(detections, labels)]

In [8]:
# Initialize sampler with distance=3 and default p=0.005
sampler = SurfaceCodeSampler(d=3, p=0.005, device=device)

# Define the error rates we want to sample from
p_values = [0.001, 0.002, 0.003, 0.004, 0.005]

# Evenly distributed weights (20% each)
p_weights = [0.2, 0.2, 0.2, 0.2, 0.2]

# Generate training data with 10,000 samples
num_samples = 10000
train_detections, train_labels, p_indices = sampler.sample(
    num_samples=num_samples,
    p_values=p_values,
    p_weights=p_weights,
    return_p_labels=True
)

# Print dataset info
print(f"\nTraining data generated:")
print(f"  Detections shape: {train_detections.shape}")
print(f"  Labels shape: {train_labels.shape}")
print(f"  Labels distribution: {train_labels.sum().item():.0f} positive / {num_samples} total")

# Verify distribution of samples across error rates
print(f"\nSamples per error rate:")
for i, p in enumerate(p_values):
    count = (p_indices == i).sum().item()
    print(f"  p={p}: {count} samples ({count/num_samples*100:.1f}%)")

SurfaceCodeSampler initialized:
  Distance: 3
  Default error rate: 0.005
  Device: cpu
  Num detectors: 24

Training data generated:
  Detections shape: torch.Size([10000, 24])
  Labels shape: torch.Size([10000])
  Labels distribution: 659 positive / 10000 total

Samples per error rate:
  p=0.001: 2000 samples (20.0%)
  p=0.002: 2000 samples (20.0%)
  p=0.003: 2000 samples (20.0%)
  p=0.004: 2000 samples (20.0%)
  p=0.005: 2000 samples (20.0%)


In [9]:
# Example: SparseGraph usage with the sampler data

# Create graph builder from the sampler's circuit
graph_builder = SparseGraph(sampler.circuit, k_neighbors=6, device=device)

# Convert a small subset to PyG graphs for demonstration
num_demo = 100
demo_graphs = graph_builder.batch_to_pyg(train_detections[:num_demo], train_labels[:num_demo])

# Show statistics
print(f"\nConverted {len(demo_graphs)} samples to sparse graphs:")

# Analyze graph statistics
node_counts = [g.x.shape[0] for g in demo_graphs]
edge_counts = [g.edge_index.shape[1] for g in demo_graphs]

print(f"  Nodes per graph: min={min(node_counts)}, max={max(node_counts)}, avg={np.mean(node_counts):.1f}")
print(f"  Edges per graph: min={min(edge_counts)}, max={max(edge_counts)}, avg={np.mean(edge_counts):.1f}")

# Show details of first non-empty graph
for i, g in enumerate(demo_graphs):
    if g.x.shape[0] > 0:
        print(f"\nExample graph (sample {i}):")
        print(f"  Num nodes (fired detectors): {g.x.shape[0]}")
        print(f"  Node features shape: {g.x.shape}")
        print(f"  Edge index shape: {g.edge_index.shape}")
        print(f"  Edge weights shape: {g.edge_attr.shape}")
        print(f"  Label: {g.y.item()}")
        print(f"  First node features [X?, Z?, d_North, d_West, d_time]:")
        print(f"    {g.x[0].tolist()}")
        break

TypeError: SparseGraph.__init__() got multiple values for argument 'k_neighbors'

In [ ]:
train_detections.shape

torch.Size([10000, 24])